In [ ]:
import os
import re
import hashlib
from datasets import load_dataset, Dataset

# ---------------------------------------------------------------------------
# 1) DROP ENTIRE FILE if it matches these (generated/vendored code, not
#    real hand-written logic worth learning from)
# ---------------------------------------------------------------------------
DROP_FILE_PATTERNS = re.compile(
    r"(created automatically by swig|don't modify this file|"
    r"do not edit|do not modify|autogenerated|auto-generated|"
    r"generated by the protocol buffer compiler|"
    r"generated from .*\.txt|generated by .*gencodec|"
    r"this file was generated|"
    r"generated by cython|# generated by |"
    r"swig_setattr|swig_getattr)",
    re.IGNORECASE,
)

# ---------------------------------------------------------------------------
# 2) STRIP long license header blocks (top-of-file boilerplate)
# ---------------------------------------------------------------------------
def strip_license_header(content):
    lines = content.split("\n")
    end = 0
    saw_license_kw = False
    for i, line in enumerate(lines[:80]):
        stripped = line.strip()
        if stripped.startswith("#"):
            if re.search(r"copyright|license", stripped, re.IGNORECASE):
                saw_license_kw = True
            end = i + 1
        elif stripped == "":
            end = i + 1
        else:
            break
    if saw_license_kw:
        return "\n".join(lines[end:]).lstrip("\n")
    return content


# ---------------------------------------------------------------------------
# 3) STRIP huge non-Python metadata blocks (Ansible DOCUMENTATION / EXAMPLES
#    / RETURN triple-quoted blobs) -- these are not Python logic.
# ---------------------------------------------------------------------------
METADATA_BLOCK_PATTERN = re.compile(
    r"^(?:DOCUMENTATION|EXAMPLES|RETURN|ANSIBLE_METADATA)\s*=\s*"
    r"('''|\"\"\")(?:.*?)\1\s*\n?",
    re.DOTALL | re.MULTILINE,
)


def strip_metadata_blocks(content):
    return METADATA_BLOCK_PATTERN.sub("", content)


# ---------------------------------------------------------------------------
# 4) REDACT email addresses anywhere in the file
# ---------------------------------------------------------------------------
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")


def redact_emails(content):
    return EMAIL_PATTERN.sub("<EMAIL>", content)


# ---------------------------------------------------------------------------
# 5) STRIP low-value boilerplate comment lines (Filename:, Author:, etc.)
#    Keep real docstrings untouched -- this only targets '#'-style lines.
# ---------------------------------------------------------------------------
BOILERPLATE_LINE_PATTERN = re.compile(
    r"^[ \t]*#[ \t]*(filename|author|created|version|date|maintainer)[ \t]*:.*\n?",
    re.IGNORECASE | re.MULTILINE,
)


def strip_boilerplate_lines(content):
    return BOILERPLATE_LINE_PATTERN.sub("", content)


# ---------------------------------------------------------------------------
# Pipeline
# ---------------------------------------------------------------------------
def clean(content):
    content = strip_license_header(content)
    content = strip_metadata_blocks(content)
    content = strip_boilerplate_lines(content)
    content = redact_emails(content)
    return content.strip()


def should_drop_file(content):
    if DROP_FILE_PATTERNS.search(content[:1000]):
        return True
    if len(content) < 100 or len(content) > 20000:
        return True
    return False


# ---------------------------------------------------------------------------
# Config -- adjust these
# ---------------------------------------------------------------------------
TARGET_COUNT = 1800000                          # how many cleaned files to collect
HF_REPO_ID = "Ananda100/python-clean-codeparrot"  # canonical name used in the paper
PRIVATE_REPO = True                             # keep True -- see licensing note below

# ---------------------------------------------------------------------------
# Run: stream, clean, dedupe, collect
# ---------------------------------------------------------------------------
ds = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True)

seen_hashes = set()
cleaned_examples = []

for example in ds:
    if example["autogenerated"]:
        continue

    raw = example["content"]
    if should_drop_file(raw):
        continue

    cleaned = clean(raw)
    if not cleaned:
        continue

    h = hashlib.sha256(cleaned.encode("utf-8", errors="ignore")).hexdigest()
    if h in seen_hashes:
        continue
    seen_hashes.add(h)

    cleaned_examples.append({"content": cleaned})

    if len(cleaned_examples) % 10_000 == 0:
        print(f"Collected {len(cleaned_examples)} cleaned files so far...")

    if len(cleaned_examples) >= TARGET_COUNT:
        break

print(f"Done cleaning. Total kept: {len(cleaned_examples)}")

# ---------------------------------------------------------------------------
# Push to Hugging Face Hub
# ---------------------------------------------------------------------------
# Requires being logged in first, e.g. in your terminal:
#   huggingface-cli login
# (paste your token there when prompted -- never hardcode it in this script)

final_ds = Dataset.from_list(cleaned_examples)


In [ ]:
final_ds.push_to_hub(
    HF_REPO_ID,
    token="YOUR_HF_TOKEN"  # never hardcode - use getpass() instead
)

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "Ananda100/python-clean-codeparrot",
    split="train",
    streaming=True  # set to False if you want to load it fully into memory
)

for i, example in enumerate(ds):
    print(f"--- Example {i+1} ---")
    print(example["content"][:800])
    print()
    if i == 9:
        break